[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/epipolar/epipolar.ipynb)

# Epipolar Geometry and Two-View Reconstruction

This example recovers the relative pose of two cameras and the 3D positions of observed landmarks from corresponding sight rays. Two cameras photograph the same scene from slightly different places. Each picture alone says only in which direction each landmark lies, not how far away it is. Together they say much more: the two sight rays toward the same landmark must cross in space, and that requirement alone fixes how the second camera is turned and in which direction it moved, and then where every landmark is. Only the overall scale is lost: a scene twice as large, seen from twice as far, gives the same pictures.

Each sight ray is a line, the join of the eye with an image point, and two lines cross when their regressive product `rays_1 & (motor >> rays_2)` vanishes. That product, a number per pair of rays, is the residual to drive to zero over the motor that places the second camera. A twist moves a line by their commutator, so the residuals' rates under a small motion are linear forms on twists, and Gauss-Newton steps are solves against the sum of their squares, a form on two twists, measured by the second camera's kinetic energy. With the pose found, a ray's squared distance to a point is the quadric `(rays & Point).scalar_norm_squared()`, and each landmark is the vertex of its two rays' quadrics summed.

In the matrix notation of computer vision the crossing condition reads as the essential matrix constraint $\mathbf r_2^{\mathsf T} E\, \mathbf r_1 = 0$ with $E = [\mathbf t]_\times R$, solved by the eight-point algorithm and a singular value decomposition into four candidate poses, followed by linear triangulation.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import show_animation
from examples.estimation.epipolar import render
from examples.estimation.epipolar.scenarios import HOUSE_EDGES, house

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Point = ga.gatype.antivector()                      # Antivector-
Line = ga.gatype.antibivector()                     # Antibivector-
Motor = ga.gatype.rotor()                           # Rotor+
Twist = ga.gatype.bivector()                        # Bivector+
# Camera 2 as a rigid body, by which a step is measured: unit masses a unit from its centre along each axis.
body = mv.point(np.concatenate([np.eye(3), -np.eye(3)]))                 # [masses] Point-

## 1. Two pictures

Camera 1 sits at the origin and looks along $+z$ at the screen one unit away; a point's image is where the sight ray through it, its join with the eye, meets the screen, `(origin & Point) ^ screen`. Camera 2 is the same camera, turned about $y$ and shifted, so it sees the landmarks moved by the inverse of its pose. Both images get a little sensor noise, a displacement within the screen.

In [ ]:
# Camera 2's true pose, turned about y and then shifted; sensor noise on the screen at unit distance, about
# 1.5 pixels on a 1000 pixel sensor, and its seed.
turn = np.radians(14.0)
shift = np.array([0.65, 0.08, 0.18])
noise, seed = 0.0015, 42

In [ ]:
landmarks = house()                                                            # [landmarks] Point-
# Camera 1 is at the world origin, looking along +z at the screen z == 1.
origin = mv.w.dual()                                                           # [] Point-
screen = mv.z - mv.w                                                           # [] Plane+
camera = (origin & Point) ^ screen                                             # [] Point- <- Point-

true_shift = (stack((mv.xw, mv.yw, mv.zw)) * shift).sum(axis=0) / 2            # [] Twist+
true_motor = (true_shift.exp() * (mv.zx * (turn / 2)).exp()).normalized()      # [] Motor+

# Each camera images the landmarks in its own frame; dividing by the weight gives the image points unit weight.
images = camera(stack([landmarks, true_motor << landmarks]))                   # [views, landmarks] Point-
images = images / (mv.w & images)                                              # [views, landmarks] Point-
screen_axes = mv.direction([[1, 0, 0], [0, 1, 0]])                             # [axes] Point-
offsets = np.random.default_rng(seed).normal(0.0, noise, size=images.shape + (2,))
measured_images = images + (screen_axes * offsets).sum(axis=-1)                # [views, landmarks] Point-
rays = (origin & measured_images).normalized()                                 # [views, landmarks] Line-

In [ ]:
print("the camera, a map from points to their images:\n", camera.coefficient_table())

## 2. The pose from crossing rays

The search starts from the right baseline without any turn. Each step solves for the twist that best zeroes the regressive products of the paired rays; the length of the baseline is the one direction the pictures cannot see, and the solve drops it, so the baseline keeps the length it started with. With the pose found at any step, each landmark is triangulated as the point nearest both its sight rays.

In [ ]:
n_iterations = 4

In [ ]:
def refine(rays_1: Line, rays_2: Line, motor: Motor) -> Motor:
    """The relative camera motor after one Gauss-Newton step, in camera 1's frame, toward crossing sight rays.

    The rays are calibrated sight rays, each in its own camera's frame. The motor needs a nonzero
    baseline; the baseline's length is not observable and stays close to the given one.
    """
    # Camera 2's kinetic energy on twists in its own frame, to measure the step by.
    kinetic = Twist & (body & body.commutator(Twist)).sum(axis=0)        # [] Scalar+ <- (Twist+, Twist+)
    # Two lines meet (are coplanar) iff their regressive product vanishes. For two lines
    # it is a number, one residual per ray pair.
    residuals = rays_1 & (motor >> rays_2)                               # [rays] Scalar-

    # A twist moves a line by their commutator, so the residual's rate of change under a
    # twist is a linear form on twists, its Jacobian.
    jacobian = rays_1 & Twist.commutator(motor >> rays_2)                # [rays] Scalar- <- Twist+

    # Gauss-Newton normal equations as forms on twists, summed over ray pairs. The residual
    # is already a number, so squaring it needs no metric: the curvature is the Jacobian
    # form times itself, and the gradient the residual times the Jacobian form.
    curvature = (jacobian * jacobian).sum(axis=0)                        # [] Scalar+ <- (Twist+, Twist+)
    descent = -(residuals * jacobian).sum(axis=0)                        # [] Scalar+ <- Twist+

    # Solve for the 5 observable degrees of freedom, measuring a step by the kinetic energy
    # of camera 2. The flattest mode is the unobservable scale, which moves camera 2 along the
    # baseline; rcond drops it by its curvature per unit energy, and the step has no
    # component along it in that energy, so near the solution the baseline keeps its length.
    energy = kinetic(motor << Twist, motor << Twist)                     # [] Scalar+ <- (Twist+, Twist+)
    step = curvature.lstsq(descent, energy, rcond=1e-4)                  # [] Twist+
    return ((step * 0.5).exp() * motor).normalized()                     # [] Motor+


def triangulate(rays_1: Line, rays_2: Line, motor: Motor) -> Point:
    """The world points triangulated from the sight rays of both cameras, in camera 1's frame, given camera 2's motor."""
    # ray & Point is the plane through a ray and an unknown point, and its squared norm is the point's
    # squared distance from the ray. The world point is the vertex of the distance quadrics summed over
    # both views. A dyad on the point's weight pins the scale without moving the vertex, so it is one solve.
    rays = stack([rays_1, motor >> rays_2])                              # [views, rays] Line-
    distances = (rays & Point).scalar_norm_squared().sum(axis=0)         # [rays] Scalar+ <- (Point-, Point-)
    weight = mv.w & Point                                                # [] Scalar+ <- Point-
    return (distances + weight * weight).solve(weight).normalized()      # [rays] Point-


def reconstruction(rays_1: Line, rays_2: Line, motor: Motor) -> Iterator[tuple[Motor, Point]]:
    """The relative camera motor and its triangulated world points at each Gauss-Newton step from an initial motor."""
    while True:
        yield motor, triangulate(rays_1, rays_2, motor)
        motor = refine(rays_1, rays_2, motor)


iterates = islice(reconstruction(rays[0], rays[1], true_shift.exp().normalized()), n_iterations + 1)
estimated_motors, reconstructed_landmarks = (stack(path) for path in zip(*iterates))  # [iterations] Motor+, [iterations, landmarks] Point-
estimated_motor = estimated_motors[-1]                                         # [] Motor+

In [ ]:
def errors(landmarks: Point, reconstructed_landmarks: Point, true_motor: Motor, estimated_motor: Motor) -> tuple[Scalar, Scalar, Scalar]:
    """The cosines between the true and estimated turns of the axis planes and between the baselines,
    and the house's RMS position error once scaled about camera 1 to the true baseline."""
    axes = stack((mv.x, mv.y, mv.z))                                          # [axes] Plane+
    alignment = (estimated_motor >> axes).normalized() | (true_motor >> axes).normalized()  # [axes] Scalar+
    origin = mv.w.dual()                                                      # [] Point-
    true_baseline = origin & (true_motor >> origin)                           # [] Line-
    estimated_baseline = origin & (estimated_motor >> origin)                 # [] Line-
    agreement = true_baseline.normalized().dot(estimated_baseline.normalized())  # [] Scalar+
    scale = true_baseline.norm() / estimated_baseline.norm()                  # [] Scalar+
    unit_landmarks = reconstructed_landmarks / (mv.w & reconstructed_landmarks)  # [landmarks] Point-
    # The difference of two unit points is a direction; its length is the norm of its dual plane.
    displacements = ((unit_landmarks * scale + origin * (1.0 - scale)) - landmarks).dual()  # [landmarks] Plane+
    return alignment, agreement, displacements.norm_squared().mean(axis=0).square_root()


alignment, agreement, rms = errors(landmarks, reconstructed_landmarks[-1], true_motor, estimated_motor)

In [ ]:
print("turn error of each axis plane, degrees:", np.degrees(np.arccos(np.minimum(alignment.to_array(), 1))))
print("baseline direction error, degrees:", np.degrees(np.arccos(np.minimum(agreement.to_array(), 1))))
print("house error once scaled to the true baseline, RMS:", rms.to_array())

## 3. Animating the reconstruction

The animation eases camera 2 from one step of the search to the next along the screw between them, and at every pose triangulates the landmarks it implies. Camera 1 is blue, and camera 2 is grey at its true pose and green at its estimated pose. The true house is grey with its landmarks joined by its edges, and the triangulated landmarks are drawn in colour along with a subset of sight rays from both cameras. Because the images do not observe the baseline's length, the estimated camera and landmarks are drawn scaled about camera 1 to the true baseline. From the initial guess without any rotation, the sight rays pass each other far apart and the triangulated points scatter; within three Gauss-Newton steps they settle onto the true landmarks up to the sensor noise.

In [ ]:
# The frames from one step to the next and held at the end, and a frame's duration in milliseconds.
n_easing, n_hold, frame_duration = 10, 16, 50

In [ ]:
def easing(rays: Line, estimated_motors: Motor, n_easing: int, n_hold: int) -> Iterator[tuple[Motor, Point]]:
    """Camera 2's motor and its triangulated world points eased from each Gauss-Newton iterate to the next along the
    interpolating screw, then held at the final pose."""
    fractions = np.arange(n_easing) / n_easing
    for start, end in zip(estimated_motors[:-1], estimated_motors[1:]):
        # The screw from one pose to the next, its log, taken a fraction of the way.
        screw = (end * start.inverse()).log()                                  # [] Twist+
        for fraction in fractions:
            motor = ((screw * fraction).exp() * start).normalized()            # [] Motor+
            yield motor, triangulate(rays[0], rays[1], motor)
    for _ in range(n_hold):
        yield estimated_motors[-1], triangulate(rays[0], rays[1], estimated_motors[-1])


show_animation(
    render.animate_reconstruction(landmarks, np.array(HOUSE_EDGES), true_motor, easing(rays, estimated_motors, n_easing, n_hold)),
    "epipolar_reconstruction",
    frame_duration,
)

In [ ]:
# checks
from examples.estimation.epipolar import core

core_iterates = islice(core.reconstruction(rays[0], rays[1], true_shift.exp().normalized()), n_iterations + 1)
core_motors, core_landmarks = (stack(path) for path in zip(*core_iterates))
np.testing.assert_allclose((estimated_motors - core_motors).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((reconstructed_landmarks - core_landmarks).kernel, 0.0, atol=1e-12)
# The noise limits the recovery: each axis plane turns within half a degree of the truth, the baseline
# direction comes back within two degrees, and the house within 10 cm RMS.
assert np.all(alignment > np.cos(np.radians(0.5)))
assert agreement > np.cos(np.radians(2.0))
assert rms < 0.1